<a href="https://colab.research.google.com/github/ihstepura/IB9AU_/blob/main/Task9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

I learned that accurate results depend on tailoring the prompting strategy to the problem. Tasks 7 and 8 taught me how important prompting is in order to get exactly what you need. Generic prompts often fail to capture the difference between complex business models, as was apparent with task 8. Last but not least, calendar math is Achilles ' heel of LLM, the default prompt did not work, and I used coding to force LLM to count different periods and not to fool itself into the trap of calendar math.

# Small Business Loan Audit — Exact Actual-Day Interest Method

This notebook audits a \$100,000 loan with **irregular payment dates** and calculates interest using the **exact actual-day method**:

$$\text{Interest} = \text{Balance} \times \frac{0.12}{365} \times \text{Days Elapsed}$$

**Key details:**
- Principal: \$100,000
- Annual rate: 12%
- Loan start: January 1, 2024
- 2024 is a **leap year** → February 29 exists and is used
- Three payments of \$5,000 each on Jan 31, Feb 29, and Mar 31

In [1]:
from datetime import date

# ──────────────────────────────────────────────
# 1. LOAN PARAMETERS
# ──────────────────────────────────────────────

principal = 100_000.00          # Initial loan amount
annual_rate = 0.12              # 12% annual interest rate
daily_rate = annual_rate / 365  # Daily interest rate (exact-day basis)

loan_start = date(2024, 1, 1)  # Loan origination date

# Payment log: list of (payment_date, payment_amount) tuples
# Note: February 29, 2024 is valid because 2024 IS a leap year.
payments = [
    (date(2024, 1, 31),  5_000.00),
    (date(2024, 2, 29),  5_000.00),   # Leap day!
    (date(2024, 3, 31),  5_000.00),
]

print(f"Loan Principal : ${principal:,.2f}")
print(f"Annual Rate    : {annual_rate:.2%}")
print(f"Daily Rate     : {daily_rate:.10f}  (0.12 / 365)")
print(f"Loan Start Date: {loan_start}")
print(f"Number of Payments: {len(payments)}")

Loan Principal : $100,000.00
Annual Rate    : 12.00%
Daily Rate     : 0.0003287671  (0.12 / 365)
Loan Start Date: 2024-01-01
Number of Payments: 3


## Verify Leap Year & February 29

Before proceeding, we explicitly confirm that 2024 is a leap year and that `date(2024, 2, 29)` is valid.

In [2]:
import calendar

# calendar.isleap() returns True if the year is a leap year
is_leap = calendar.isleap(2024)
print(f"Is 2024 a leap year? {is_leap}")

# Directly construct Feb 29 — this would raise ValueError if 2024 were not a leap year
feb_29 = date(2024, 2, 29)
print(f"February 29, 2024 successfully created: {feb_29}")
print(f"Day of week: {feb_29.strftime('%A')}")

Is 2024 a leap year? True
February 29, 2024 successfully created: 2024-02-29
Day of week: Thursday


## Amortization Calculation

For each payment period we:
1. Count the **exact calendar days** between the previous date and the current payment date.
2. Compute accrued interest: `Balance × (0.12 / 365) × Days`.
3. Add interest to the balance, then subtract the payment.
4. Record everything in a table row.

In [3]:
# ──────────────────────────────────────────────
# 2. BUILD THE AMORTIZATION SCHEDULE
# ──────────────────────────────────────────────

schedule = []          # Will hold one dict per payment period
balance = principal    # Running balance starts at the principal
prev_date = loan_start # Track the start of each period

for pay_date, payment in payments:
    # --- Days elapsed: exact calendar-day difference ---
    # (date subtraction in Python gives a timedelta; .days is the integer count)
    days_elapsed = (pay_date - prev_date).days

    # --- Accrued interest for this exact period ---
    # Formula: Balance * daily_rate * days_elapsed
    interest = balance * daily_rate * days_elapsed

    # --- Update the balance ---
    beginning_balance = balance
    balance += interest    # Add accrued interest
    balance -= payment     # Subtract the payment

    # --- Store the row ---
    schedule.append({
        "period_start": prev_date,
        "period_end": pay_date,
        "days_elapsed": days_elapsed,
        "beginning_balance": beginning_balance,
        "interest_accrued": interest,
        "payment": payment,
        "ending_balance": balance,
    })

    # Advance the period start to current payment date
    prev_date = pay_date

print(f"Schedule built — {len(schedule)} periods computed.")

Schedule built — 3 periods computed.


## Amortization Table

Displaying the results in a clear, formatted table with all monetary values rounded to 2 decimal places.

In [4]:
# ──────────────────────────────────────────────
# 3. DISPLAY THE AMORTIZATION TABLE
# ──────────────────────────────────────────────

# Header
print("=" * 110)
print(f"{'Period Start':<15} {'Period End':<15} {'Days':>5} {'Beginning Bal':>16} "
      f"{'Interest':>14} {'Payment':>12} {'Ending Bal':>16}")
print("=" * 110)

# Rows
for row in schedule:
    print(
        f"{str(row['period_start']):<15} "
        f"{str(row['period_end']):<15} "
        f"{row['days_elapsed']:>5} "
        f"${row['beginning_balance']:>14,.2f} "
        f"${row['interest_accrued']:>12,.2f} "
        f"${row['payment']:>10,.2f} "
        f"${row['ending_balance']:>14,.2f}"
    )

print("=" * 110)

# Final balance
final_balance = schedule[-1]["ending_balance"]
print(f"\n>>> FINAL ENDING BALANCE after {payments[-1][0]} payment: ${final_balance:,.2f}")

Period Start    Period End       Days    Beginning Bal       Interest      Payment       Ending Bal
2024-01-01      2024-01-31         30 $    100,000.00 $      986.30 $  5,000.00 $     95,986.30
2024-01-31      2024-02-29         29 $     95,986.30 $      915.16 $  5,000.00 $     91,901.46
2024-02-29      2024-03-31         31 $     91,901.46 $      936.64 $  5,000.00 $     87,838.10

>>> FINAL ENDING BALANCE after 2024-03-31 payment: $87,838.10


## Pandas DataFrame View

An alternative, cleaner view using `pandas` for easier inspection and export.

In [5]:
import pandas as pd

# ──────────────────────────────────────────────
# 4. PANDAS DATAFRAME VIEW
# ──────────────────────────────────────────────

df = pd.DataFrame(schedule)

# Rename columns for a polished display
df.columns = [
    "Period Start", "Period End", "Days Elapsed",
    "Beginning Balance", "Interest Accrued",
    "Payment", "Ending Balance"
]

# Format monetary columns to 2 decimal places with dollar signs
money_cols = ["Beginning Balance", "Interest Accrued", "Payment", "Ending Balance"]
styled = df.style.format(
    {col: "${:,.2f}" for col in money_cols}
).set_caption("Loan Amortization Schedule — Exact Actual-Day Method")

styled

,Period Start,Period End,Days Elapsed,Beginning Balance,Interest Accrued,Payment,Ending Balance
0,2024-01-01,2024-01-31,30,"$100,000.00",$986.30,"$5,000.00","$95,986.30"
1,2024-01-31,2024-02-29,29,"$95,986.30",$915.16,"$5,000.00","$91,901.46"
2,2024-02-29,2024-03-31,31,"$91,901.46",$936.64,"$5,000.00","$87,838.10"


## Verification: Day-Count Accuracy

We manually verify that the exact day counts between each pair of dates are correct.

In [6]:
# ──────────────────────────────────────────────
# 5. VERIFICATION OF DAY COUNTS
# ──────────────────────────────────────────────

print("Day-Count Verification")
print("=" * 60)

# Period 1: Jan 1 → Jan 31
d1 = (date(2024, 1, 31) - date(2024, 1, 1)).days
print(f"\n1) Jan 1, 2024 → Jan 31, 2024")
print(f"   Days elapsed: {d1}")
print(f"   Explanation : January has 31 days. From day 1 to day 31")
print(f"                 is 30 days elapsed (we count the gap, not")
print(f"                 the endpoints). 31 - 1 = 30. ✓")

# Period 2: Jan 31 → Feb 29
d2 = (date(2024, 2, 29) - date(2024, 1, 31)).days
print(f"\n2) Jan 31, 2024 → Feb 29, 2024")
print(f"   Days elapsed: {d2}")
print(f"   Explanation : From Jan 31 to Feb 29. February 2024 has")
print(f"                 29 days (leap year!). Jan 31 → Feb 29 spans")
print(f"                 the entire month of February except its last")
print(f"                 day as an endpoint = 29 days. ✓")

# Period 3: Feb 29 → Mar 31
d3 = (date(2024, 3, 31) - date(2024, 2, 29)).days
print(f"\n3) Feb 29, 2024 → Mar 31, 2024")
print(f"   Days elapsed: {d3}")
print(f"   Explanation : From Feb 29 to Mar 31. There is 1 remaining")
print(f"                 day in February (none, since Feb 29 is the")
print(f"                 last day) plus all 31 days of March = 31 days.")
print(f"                 Alternatively: Mar 31 is day 91 of 2024,")
print(f"                 Feb 29 is day 60. 91 - 60 = 31. ✓")

# Total days from loan start to final payment
total = d1 + d2 + d3
print(f"\nTotal days from Jan 1 → Mar 31: {total}")
print(f"Cross-check: (Mar 31 - Jan 1) = {(date(2024,3,31) - date(2024,1,1)).days} days ✓")

Day-Count Verification

1) Jan 1, 2024 → Jan 31, 2024
   Days elapsed: 30
   Explanation : January has 31 days. From day 1 to day 31
                 is 30 days elapsed (we count the gap, not
                 the endpoints). 31 - 1 = 30. ✓

2) Jan 31, 2024 → Feb 29, 2024
   Days elapsed: 29
   Explanation : From Jan 31 to Feb 29. February 2024 has
                 29 days (leap year!). Jan 31 → Feb 29 spans
                 the entire month of February except its last
                 day as an endpoint = 29 days. ✓

3) Feb 29, 2024 → Mar 31, 2024
   Days elapsed: 31
   Explanation : From Feb 29 to Mar 31. There is 1 remaining
                 day in February (none, since Feb 29 is the
                 last day) plus all 31 days of March = 31 days.
                 Alternatively: Mar 31 is day 91 of 2024,
                 Feb 29 is day 60. 91 - 60 = 31. ✓

Total days from Jan 1 → Mar 31: 90
Cross-check: (Mar 31 - Jan 1) = 90 days ✓


## Manual Interest Verification

Step-by-step recalculation to confirm each period's interest and ending balance.

In [7]:
# ──────────────────────────────────────────────
# 6. STEP-BY-STEP MANUAL VERIFICATION
# ──────────────────────────────────────────────

print("Manual Interest & Balance Verification")
print("=" * 60)

# Period 1
b0 = 100_000.00
i1 = b0 * (0.12 / 365) * 30
b1 = b0 + i1 - 5_000.00
print(f"\nPeriod 1 (Jan 1 → Jan 31, 30 days):")
print(f"  Beginning Balance : ${b0:,.2f}")
print(f"  Interest          : $100,000 × (0.12/365) × 30 = ${i1:,.2f}")
print(f"  Payment           : $5,000.00")
print(f"  Ending Balance    : ${b1:,.2f}")

# Period 2
i2 = b1 * (0.12 / 365) * 29
b2 = b1 + i2 - 5_000.00
print(f"\nPeriod 2 (Jan 31 → Feb 29, 29 days):")
print(f"  Beginning Balance : ${b1:,.2f}")
print(f"  Interest          : ${b1:,.2f} × (0.12/365) × 29 = ${i2:,.2f}")
print(f"  Payment           : $5,000.00")
print(f"  Ending Balance    : ${b2:,.2f}")

# Period 3
i3 = b2 * (0.12 / 365) * 31
b3 = b2 + i3 - 5_000.00
print(f"\nPeriod 3 (Feb 29 → Mar 31, 31 days):")
print(f"  Beginning Balance : ${b2:,.2f}")
print(f"  Interest          : ${b2:,.2f} × (0.12/365) × 31 = ${i3:,.2f}")
print(f"  Payment           : $5,000.00")
print(f"  Ending Balance    : ${b3:,.2f}")

# Summary
total_interest = i1 + i2 + i3
total_payments = 15_000.00
print(f"\n{'─' * 60}")
print(f"Total Interest Accrued : ${total_interest:,.2f}")
print(f"Total Payments Made    : ${total_payments:,.2f}")
print(f"Final Ending Balance   : ${b3:,.2f}")
print(f"\nFormula check: $100,000 + ${total_interest:,.2f} − ${total_payments:,.2f} = ${b3:,.2f} ✓")

Manual Interest & Balance Verification

Period 1 (Jan 1 → Jan 31, 30 days):
  Beginning Balance : $100,000.00
  Interest          : $100,000 × (0.12/365) × 30 = $986.30
  Payment           : $5,000.00
  Ending Balance    : $95,986.30

Period 2 (Jan 31 → Feb 29, 29 days):
  Beginning Balance : $95,986.30
  Interest          : $95,986.30 × (0.12/365) × 29 = $915.16
  Payment           : $5,000.00
  Ending Balance    : $91,901.46

Period 3 (Feb 29 → Mar 31, 31 days):
  Beginning Balance : $91,901.46
  Interest          : $91,901.46 × (0.12/365) × 31 = $936.64
  Payment           : $5,000.00
  Ending Balance    : $87,838.10

────────────────────────────────────────────────────────────
Total Interest Accrued : $2,838.10
Total Payments Made    : $15,000.00
Final Ending Balance   : $87,838.10

Formula check: $100,000 + $2,838.10 − $15,000.00 = $87,838.10 ✓
